# Bài 2.1 — Linear Regression từ đầu

Mục tiêu: tạo dữ liệu tuyến tính có nhiễu, train bằng sklearn và tự cài đặt batch Gradient Descent. Không dùng sklearn để train phần Gradient Descent.

## Bước 1 — Import và cấu hình experiment

Seed, train/test split, số epoch và learning rate đều được ghi rõ để chạy lại cho cùng kết quả.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

SEED = 42
TRUE_W = 3.5
TRUE_B = 20.0
NOISE_STD = 20.0
N_SAMPLES = 200
TEST_SIZE = 0.2
EXPERIMENT_EPOCHS = 1_000
FINAL_EPOCHS = 100_000
LEARNING_RATES = [0.000001, 0.00001, 0.0001, 0.001]
BEST_LEARNING_RATE = 0.0001

## Bước 2 — Sinh dữ liệu và chia train/test

Dữ liệu tuân theo $y = 3.5x + 20 + noise$. Nhiễu Gaussian khiến các điểm không nằm hoàn toàn trên một đường thẳng.

In [ ]:
np.random.seed(SEED)
x = np.linspace(0, 100, N_SAMPLES)
y = TRUE_W * x + TRUE_B + np.random.normal(0, NOISE_STD, N_SAMPLES)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=TEST_SIZE, random_state=SEED
)

print(f'Total samples: {len(x)}')
print(f'Train samples: {len(x_train)}')
print(f'Test samples:  {len(x_test)}')

## Bước 3 — Baseline bằng sklearn

Đây là mốc để kiểm chứng phần tự cài. `reshape(-1, 1)` đổi mảng `x` một chiều thành ma trận có một feature mà sklearn yêu cầu.

In [ ]:
sk_model = LinearRegression()
sk_model.fit(x_train.reshape(-1, 1), y_train)

sk_train_pred = sk_model.predict(x_train.reshape(-1, 1))
sk_test_pred = sk_model.predict(x_test.reshape(-1, 1))

print(f'w = {sk_model.coef_[0]:.6f}')
print(f'b = {sk_model.intercept_:.6f}')
print(f'Train MSE = {mean_squared_error(y_train, sk_train_pred):.6f}')
print(f'Test MSE  = {mean_squared_error(y_test, sk_test_pred):.6f}')

## Bước 4 — Tự cài đặt Linear Regression bằng Gradient Descent

Bốn hàm bên dưới là phần cài đặt chính. `compute_gradients` tính độ dốc của MSE theo `w` và `b`; `train` dùng độ dốc đó để cập nhật tham số sau mỗi epoch.

In [ ]:
def predict(x, w, b):
    return w * x + b


def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def compute_gradients(x, y_true, w, b):
    errors = predict(x, w, b) - y_true
    n = len(x)
    dw = (2 / n) * np.sum(x * errors)
    db = (2 / n) * np.sum(errors)
    return dw, db


def train(x, y_true, learning_rate, epochs, w=0.0, b=0.0):
    loss_history = [mse(y_true, predict(x, w, b))]
    diverged = False

    for _ in range(epochs):
        dw, db = compute_gradients(x, y_true, w, b)
        w -= learning_rate * dw
        b -= learning_rate * db
        current_loss = mse(y_true, predict(x, w, b))
        loss_history.append(current_loss)

        if not np.isfinite(current_loss) or current_loss > 1e100:
            diverged = True
            break

    return w, b, np.array(loss_history), diverged

## Bước 5 — Thử bốn learning rate trong 1.000 epoch

Mỗi lần thử đều bắt đầu từ `w=0`, `b=0`, nên khác biệt kết quả chỉ đến từ learning rate.

In [ ]:
experiments = []
for learning_rate in LEARNING_RATES:
    w, b, loss_history, diverged = train(
        x_train, y_train, learning_rate, EXPERIMENT_EPOCHS
    )
    test_mse = np.nan if diverged else mse(y_test, predict(x_test, w, b))
    experiments.append({
        'learning_rate': learning_rate,
        'w': w,
        'b': b,
        'train_mse': loss_history[-1],
        'test_mse': test_mse,
        'loss_history': loss_history,
        'diverged': diverged,
    })

print('lr\t\tw\t\tb\t\ttrain_mse\ttest_mse\tstatus')
for result in experiments:
    status = 'diverged' if result['diverged'] else 'completed'
    print(f"{result['learning_rate']:.6g}\t{result['w']:.6f}\t{result['b']:.6f}\t"
          f"{result['train_mse']:.6f}\t{result['test_mse']:.6f}\t{status}")

## Bước 6 — Train cấu hình ổn định đến khi hội tụ

1.000 epoch là yêu cầu tối thiểu để so sánh learning rate. Vì `b` học chậm trên dữ liệu `x` có thang đo 0–100, chạy tiếp `lr=0.0001` đến 100.000 epoch để so sánh công bằng với sklearn.

In [ ]:
gd_w, gd_b, gd_loss, gd_diverged = train(
    x_train, y_train, BEST_LEARNING_RATE, FINAL_EPOCHS
)
gd_train_mse = mse(y_train, predict(x_train, gd_w, gd_b))
gd_test_mse = mse(y_test, predict(x_test, gd_w, gd_b))

print(f'Diverged: {gd_diverged}')
print(f'w = {gd_w:.6f}')
print(f'b = {gd_b:.6f}')
print(f'Train MSE = {gd_train_mse:.6f}')
print(f'Test MSE  = {gd_test_mse:.6f}')
print(f'Difference from sklearn w: {abs(gd_w - sk_model.coef_[0]):.6f}')
print(f'Difference from sklearn b: {abs(gd_b - sk_model.intercept_):.6f}')

## Bước 7 — Vẽ dữ liệu, đường hồi quy và loss

Đường sklearn và Gradient Descent gần chồng lên nhau khi Gradient Descent hội tụ. Biểu đồ loss dùng thang log vì learning rate 0,001 diverge.

In [ ]:
line_x = np.linspace(0, 100, 300)

plt.figure(figsize=(10, 6))
plt.scatter(x_train, y_train, label='Train', alpha=0.7)
plt.scatter(x_test, y_test, label='Test', alpha=0.7)
plt.plot(line_x, predict(line_x, sk_model.coef_[0], sk_model.intercept_),
         label='sklearn LinearRegression', linewidth=2.5)
plt.plot(line_x, predict(line_x, gd_w, gd_b),
         label='Gradient Descent (lr=0.0001)', linewidth=2.5, linestyle='--')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Linear Regression: data and fitted lines')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

fig, (ax_stable, ax_diverge) = plt.subplots(1, 2, figsize=(13, 5.5))
for result in experiments:
    safe_loss = np.clip(result['loss_history'], 1e-12, 1e100)
    label = f"lr={result['learning_rate']:g}"
    if result['diverged']:
        ax_diverge.plot(safe_loss, color='#d62728', label=label)
    else:
        ax_stable.plot(safe_loss, label=label)

for ax, title in [(ax_stable, 'Stable learning rates'),
                  (ax_diverge, 'Learning rate that diverges')]:
    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train MSE (log scale)')
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle('Gradient Descent loss by learning rate')
fig.tight_layout()
plt.show()

## Kết luận

- `lr=0.001` diverge vì bước update quá lớn.
- `lr=0.000001` và `0.00001` ổn định nhưng học quá chậm trong 1.000 epoch.
- `lr=0.0001` là lựa chọn tốt nhất trong bốn giá trị thử; sau 100.000 epoch, kết quả gần như trùng sklearn.
- Hệ số không khớp hoàn toàn với 3,5 và 20 vì dataset có Gaussian noise và chỉ 80% sample được train.